# RBF Kernel SVM — K49-MNIST Hiragana Classification

## Why RBF after Linear SVM?

Linear SVM achieved **80.07% test accuracy** but consistently showed a **~11% val-test gap**, indicating the linear decision boundary cannot generalize across different writer styles in the test set.

The **RBF (Radial Basis Function) kernel** addresses this by implicitly mapping features into infinite-dimensional space:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp\left(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2\right)$$

This allows the model to learn **curved, non-linear decision boundaries** that can better separate visually similar characters written by different people.

| | Linear SVM | RBF SVM |
|---|---|---|
| Decision boundary | Flat hyperplane | Curved, non-linear |
| Parameters | C | C + γ (gamma) |
| Training speed | Fast (LibLinear) | Slower (O(n²) kernel) |
| Inference speed | Very fast | Slower (support vectors) |
| Expected accuracy | ~80% | Higher |

### Key new parameter: γ (gamma)
- **Large γ** → each training point has very local influence → tight, complex boundaries → overfitting risk
- **Small γ** → each point influences a wide area → smoother boundaries → better generalization
- `gamma='scale'` (default) sets γ = 1 / (n_features × X.var()) — a good data-driven starting point

---
### ⚠️ Important: Dataset Size
Full `SVC` with RBF kernel is **O(n²) in memory and O(n³) in training time** — infeasible on 232,365 samples. We use two strategies:
1. Train on a **stratified subset** (30,000–50,000 samples) for CV and final model
2. Use `LinearSVC` HOG features already saved — no re-extraction needed

## 1. Import Libraries

In [ ]:
import os
import time
import threading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.frozen import FrozenEstimator
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, RandomizedSearchCV
)
from sklearn.utils import class_weight, resample
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)
from scipy.stats import loguniform

print('✅ All libraries imported successfully.')

## 2. Load Preprocessed HOG Features

We reuse the float32 HOG features saved from the Linear SVM v3 notebook — no re-extraction needed.

In [ ]:
X_train_hog = np.load('../models/v3_X_train_hog.npy')
X_test_hog  = np.load('../models/v3_X_test_hog.npy')
y_train     = np.load('../models/v3_y_tr.npy')
y_test      = np.load('../models/v3_y_test.npy')

# Reconstruct full y_train from saved splits
y_tr  = np.load('../models/v3_y_tr.npy')
y_val = np.load('../models/v3_y_val.npy')
y_train_full = np.concatenate([y_tr, y_val])

classmap     = pd.read_csv('../data/k49_classmap.csv')
target_names = [str(classmap.iloc[i]['char']) for i in range(len(classmap))]

print(f'Train HOG : {X_train_hog.shape} ({X_train_hog.dtype})')
print(f'Test HOG  : {X_test_hog.shape} ({X_test_hog.dtype})')
print(f'Features  : {X_train_hog.shape[1]} dims')
print('✅ HOG features loaded.')

## 3. Create Stratified Training Subset

Full RBF SVC on 232,365 samples would require ~400 GB RAM and days of training. We use a **stratified 30,000-sample subset** for CV tuning and a **50,000-sample subset** for the final model — large enough to cover all 49 classes with good representation while remaining computationally feasible.

| Dataset size | Approx. train time | Approx. RAM |
|---|---|---|
| 232,365 (full) | Days | ~400 GB |
| 50,000 | 30–60 min | ~10 GB |
| 30,000 | 10–20 min | ~5 GB |
| **30,000 (CV) / 50,000 (final)** | **Feasible** | **Feasible** |

In [ ]:
CV_SUBSET_SIZE    = 30_000   # used for hyperparameter search
FINAL_SUBSET_SIZE = 50_000   # used for final model training

# Stratified subset for CV
X_cv, y_cv = resample(
    X_train_hog, y_train_full,
    n_samples=CV_SUBSET_SIZE,
    random_state=42,
    stratify=y_train_full
)

# Stratified subset for final model
X_final, y_final = resample(
    X_train_hog, y_train_full,
    n_samples=FINAL_SUBSET_SIZE,
    random_state=42,
    stratify=y_train_full
)

# Validation split from final subset (for calibration)
X_tr_f, X_val_f, y_tr_f, y_val_f = train_test_split(
    X_final, y_final,
    test_size=0.2, random_state=42, stratify=y_final
)

print(f'CV subset       : {X_cv.shape[0]:,} samples')
print(f'Final train set : {X_tr_f.shape[0]:,} samples')
print(f'Calibration set : {X_val_f.shape[0]:,} samples')
print(f'Test set        : {X_test_hog.shape[0]:,} samples')

## 4. Class Weights

In [ ]:
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_full),
    y=y_train_full
)
class_weights_dict = dict(enumerate(weights))

print(f'Total classes             : {len(class_weights_dict)}')
print(f'Min weight (common class) : {min(weights):.4f}')
print(f'Max weight (rare class)   : {max(weights):.4f}')
print('✅ Class weights ready.')

## 5. Hyperparameter Tuning — RandomizedSearchCV

RBF SVM has **two parameters** to tune jointly:
- **C** — regularization (same as Linear SVM)
- **γ (gamma)** — kernel bandwidth (new for RBF)

Because the 2D grid `C × γ` is expensive, we use **RandomizedSearchCV** instead of GridSearchCV. It samples a fixed number of random combinations from continuous log-uniform distributions, giving good coverage with far fewer evaluations.

Log-uniform distributions are used because both C and γ are best searched on a log scale — the difference between 0.001 and 0.01 is as important as between 1 and 10.

⚠️ With `n_iter=20` and `cv=3` this trains **60 models** on 30,000 samples. Expect **30–60 minutes**.

In [ ]:
param_dist = {
    'C'    : loguniform(0.1, 100),    # log-uniform search: 0.1 → 100
    'gamma': loguniform(1e-4, 1e-1),  # log-uniform search: 0.0001 → 0.1
}

base_svc = SVC(
    kernel='rbf',
    class_weight=class_weights_dict,
    probability=False,   # keep False during search for speed
    random_state=42
)

search = RandomizedSearchCV(
    estimator=base_svc,
    param_distributions=param_dist,
    n_iter=20,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=2,
    refit=False   # we'll refit manually on the larger subset
)

print(f'⏳ RandomizedSearchCV: 20 iterations x 3 folds on {CV_SUBSET_SIZE:,} samples...')
t0 = time.time()
search.fit(X_cv, y_cv)
elapsed = time.time() - t0

best_C     = search.best_params_['C']
best_gamma = search.best_params_['gamma']

print(f'\n✅ Search complete in {elapsed/60:.1f} min')
print(f'🏆 Best C     = {best_C:.6f}')
print(f'🏆 Best gamma = {best_gamma:.6f}')
print(f'🏆 Best CV accuracy = {search.best_score_:.4f}')

In [ ]:
# --- Visualize search results ---
results_df = pd.DataFrame(search.cv_results_)
results_df = results_df.sort_values('mean_test_score', ascending=False)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.scatter(np.log10(results_df['param_C'].astype(float)),
            results_df['mean_test_score'],
            c=results_df['mean_test_score'], cmap='viridis', s=60)
plt.axvline(np.log10(best_C), color='red', linestyle='--', label=f'Best C={best_C:.4f}')
plt.xlabel('log10(C)')
plt.ylabel('Mean CV Accuracy')
plt.title('Accuracy vs C')
plt.legend(fontsize=8)

plt.subplot(1, 2, 2)
plt.scatter(np.log10(results_df['param_gamma'].astype(float)),
            results_df['mean_test_score'],
            c=results_df['mean_test_score'], cmap='viridis', s=60)
plt.axvline(np.log10(best_gamma), color='red', linestyle='--', label=f'Best γ={best_gamma:.5f}')
plt.xlabel('log10(gamma)')
plt.ylabel('Mean CV Accuracy')
plt.title('Accuracy vs Gamma')
plt.legend(fontsize=8)

plt.suptitle('RandomizedSearchCV Results — RBF SVM', fontsize=12)
plt.tight_layout()
plt.show()

print('\nTop 5 parameter combinations:')
print(results_df[['param_C', 'param_gamma', 'mean_test_score', 'std_test_score']].head())

## 6. Train Final RBF SVM

We retrain with the best C and γ on the larger **50,000-sample subset**.  
`probability=True` is enabled here so the model can output calibrated probabilities for the similarity score feature — this makes training slightly slower but is required for `predict_proba`.

⚠️ Expect **20–40 minutes** on 40,000 training samples.

In [ ]:
final_rbf = SVC(
    kernel='rbf',
    C=best_C,
    gamma=best_gamma,
    class_weight=class_weights_dict,
    probability=True,   # enables predict_proba for similarity scores
    random_state=42
)

print(f'⏳ Training final RBF SVM (C={best_C:.4f}, γ={best_gamma:.5f})')
print(f'   on {X_tr_f.shape[0]:,} samples...')

t0   = time.time()
done = threading.Event()

def train():
    final_rbf.fit(X_tr_f, y_tr_f)
    done.set()

thread = threading.Thread(target=train)
thread.start()

with tqdm(desc='🏋️  Training RBF SVM', unit='iter',
          bar_format='{desc} | {elapsed} elapsed {postfix}') as pbar:
    while not done.is_set():
        pbar.update(0)
        done.wait(timeout=0.5)

thread.join()
train_time = time.time() - t0

n_sv = sum(final_rbf.n_support_)
print(f'\n✅ Training complete in {train_time/60:.1f} min')
print(f'   Support vectors: {n_sv:,} ({n_sv/X_tr_f.shape[0]*100:.1f}% of training set)')

## 7. Save Checkpoint

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(final_rbf, '../models/rbf_svm.pkl')
np.save('../models/rbf_X_val.npy', X_val_f)
np.save('../models/rbf_y_val.npy', y_val_f)
print('✅ Model and calibration data saved.')

### Quick Reload

In [ ]:
# Uncomment to reload on subsequent sessions
# final_rbf = joblib.load('../models/rbf_svm.pkl')
# X_val_f   = np.load('../models/rbf_X_val.npy')
# y_val_f   = np.load('../models/rbf_y_val.npy')
# best_C, best_gamma = final_rbf.C, final_rbf.gamma
# print(f'Loaded RBF SVM — C={best_C:.4f}, gamma={best_gamma:.5f}')

## 8. Evaluation on Validation Set

In [ ]:
t0 = time.time()
y_val_pred     = final_rbf.predict(X_val_f)
val_infer_time = (time.time() - t0) / len(y_val_f) * 1000

val_acc = accuracy_score(y_val_f, y_val_pred)
val_f1  = f1_score(y_val_f, y_val_pred, average='macro')

print('=' * 40)
print('  Validation Set Results')
print('=' * 40)
print(f'  Top-1 Accuracy  : {val_acc:.4f} ({val_acc*100:.2f}%)')
print(f'  Macro F1-Score  : {val_f1:.4f}')
print(f'  Inference Time  : {val_infer_time:.4f} ms/sample')
print('=' * 40)

## 9. Evaluation on Official Test Set

In [ ]:
print('⏳ Predicting on test set (may take a few minutes with RBF kernel)...')
t0 = time.time()
y_test_pred     = final_rbf.predict(X_test_hog)
test_infer_time = (time.time() - t0) / len(y_test) * 1000

test_acc = accuracy_score(y_test, y_test_pred)
test_f1  = f1_score(y_test, y_test_pred, average='macro')

print('=' * 40)
print('  Official Test Set Results')
print('=' * 40)
print(f'  Top-1 Accuracy  : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'  Macro F1-Score  : {test_f1:.4f}')
print(f'  Inference Time  : {test_infer_time:.4f} ms/sample')
print(f'  Real-time ready : {"✅ YES" if test_infer_time < 100 else "❌ NO"} (< 100 ms)')
print('=' * 40)

gap = val_acc - test_acc
print(f'\n  Val-Test gap : {gap*100:.2f}% (Linear SVM v3: ~11%)')

## 10. Linear SVM vs RBF SVM — Full Comparison

In [ ]:
# Fill in Linear SVM v3 results
lin_val_acc  = 0.9115
lin_test_acc = 0.8007
lin_test_f1  = 0.7821
lin_gap      = lin_val_acc - lin_test_acc
lin_infer    = 0.0071
lin_train_s  = 351.4

rbf_gap = val_acc - test_acc

print('=' * 62)
print(f'{"Metric":<28} {"Linear SVM v3":>14} {"RBF SVM":>12}')
print('=' * 62)
print(f'{"Val Accuracy":<28} {lin_val_acc*100:>13.2f}% {val_acc*100:>11.2f}%')
print(f'{"Test Accuracy":<28} {lin_test_acc*100:>13.2f}% {test_acc*100:>11.2f}%')
print(f'{"Test Macro F1":<28} {lin_test_f1:>14.4f} {test_f1:>12.4f}')
print(f'{"Val-Test Gap":<28} {lin_gap*100:>13.2f}% {rbf_gap*100:>11.2f}%')
print(f'{"Inference time (ms)":<28} {lin_infer:>14.4f} {test_infer_time:>12.4f}')
print(f'{"Real-time (<100ms)":<28} {"✅ YES":>14} {"✅ YES" if test_infer_time < 100 else "❌ NO":>12}')
print(f'{"Train time":<28} {lin_train_s:>12.1f}s {train_time:>11.1f}s')
print(f'{"Training samples":<28} {"232,365":>14} {X_tr_f.shape[0]:>12,}')
print('=' * 62)

## 11. Per-Class Classification Report

In [ ]:
print(classification_report(y_test, y_test_pred, target_names=target_names))

## 12. Confusion Matrix

In [ ]:
cm      = confusion_matrix(y_test, y_test_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(18, 15))
sns.heatmap(cm_norm, annot=False, fmt='.2f', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.title('Normalized Confusion Matrix — RBF SVM', fontsize=14)
plt.xlabel('Predicted Character', fontsize=12)
plt.ylabel('True Character', fontsize=12)
plt.xticks(fontsize=8, rotation=45)
plt.yticks(fontsize=8, rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
confused_pairs = [
    (cm_norm[i, j], target_names[i], target_names[j])
    for i in range(len(target_names))
    for j in range(len(target_names))
    if i != j
]
confused_pairs.sort(reverse=True)

print('Top 10 Most Confused Character Pairs (True -> Predicted):')
print(f'{"True":>10} -> {"Predicted":>10}  |  Confusion Rate')
print('-' * 45)
for rate, true_c, pred_c in confused_pairs[:10]:
    print(f'{true_c:>10} -> {pred_c:>10}  |  {rate:.4f} ({rate*100:.2f}%)')

## 13. Probability Similarity Scores

Since `probability=True` was set during training, `predict_proba` is available directly — no separate calibration step needed for RBF SVM.

In [ ]:
sample_idx = 42

sample_hog = X_test_hog[sample_idx].reshape(1, -1)
true_label = y_test[sample_idx]
proba      = final_rbf.predict_proba(sample_hog)[0]

top5_idx  = np.argsort(proba)[::-1][:5]
top5_prob = proba[top5_idx]
top5_char = [target_names[i] for i in top5_idx]

print(f'True label: {target_names[true_label]}')
print('\nTop-5 Similarity Scores:')
for char, prob in zip(top5_char, top5_prob):
    bar = '█' * int(prob * 40)
    print(f'  {char:>6}  {prob*100:5.1f}%  {bar}')

# Load raw test images for display
import io
test_imgs  = np.load('../data/k49-test-imgs.npz')
X_test_raw = test_imgs['arr_0']

plt.figure(figsize=(3, 3))
plt.imshow(X_test_raw[sample_idx], cmap='gray')
plt.title(f'True: {target_names[true_label]}')
plt.axis('off')
plt.show()

## 14. Deployment Summary

In [ ]:
import sys
model_size_mb = sys.getsizeof(joblib.dump(final_rbf, '../models/rbf_svm.pkl')) / (1024**2)
model_file_mb = os.path.getsize('../models/rbf_svm.pkl') / (1024**2)

print('=' * 50)
print('  RBF SVM — Deployment Summary')
print('=' * 50)
print(f'  Kernel              : RBF')
print(f'  Best C              : {best_C:.6f}')
print(f'  Best gamma          : {best_gamma:.6f}')
print(f'  Support vectors     : {sum(final_rbf.n_support_):,}')
print(f'  Model file size     : {model_file_mb:.1f} MB')
print(f'  Train time          : {train_time/60:.1f} min')
print(f'  Validation accuracy : {val_acc*100:.2f}%')
print(f'  Test accuracy       : {test_acc*100:.2f}%')
print(f'  Test macro F1       : {test_f1:.4f}')
print(f'  Inference time      : {test_infer_time:.4f} ms/sample')
print(f'  Real-time ready     : {"✅ YES" if test_infer_time < 100 else "❌ NO"}')
print('=' * 50)

## 15. Save Final Model

In [ ]:
joblib.dump(final_rbf, '../models/rbf_svm_final.pkl')
print('✅ RBF SVM saved to ../models/rbf_svm_final.pkl')

---
## Summary

| | Linear SVM v3 | RBF SVM |
|---|---|---|
| **Test Accuracy** | 80.07% | (fill after run) |
| **Macro F1** | 0.7821 | (fill after run) |
| **Val-Test Gap** | ~11% | (fill after run) |
| **Inference** | 0.007 ms | (fill after run) |
| **Train time** | 351s | (fill after run) |
| **Training samples** | 232,365 | 40,000 |
| **Model size** | 0.48 MB | larger (support vectors) |

### Deployment recommendation
- If RBF test accuracy is **significantly higher** with a **smaller gap** → RBF is the better model, accept the inference tradeoff if it stays under 100ms
- If RBF gap is still large or inference exceeds 100ms → Linear SVM v3 remains the deployment candidate on the grounds of speed and simplicity
- Either way, both results feed into the final model comparison section of the report